# *Nonlinear Arterial Hemodynamics*
## Chapter 3 companion — Womersley Flow as the Classical Reference State

This notebook is the computational companion to Chapter 3. It reconstructs the rigid, straight, axisymmetric Womersley reference state and uses VascuQuest/PWDB waveforms to place that reference model in a physiological virtual-population context.

The book nomenclature governs every reader-facing quantity.

The notebook admits only the mechanisms admitted by Chapter 3: incompressible Newtonian flow, scalar viscosity, rigid straight circular geometry, axisymmetry, no swirl, fully developed axial motion, and linear harmonic response.

It deliberately withholds constitutive anisotropy, geometry-sensitive modulation, wall motion, fluid–structure coupling, nonlinear harmonic transfer, and second-order mean transport.

**Execution:** a clean Google Colab runtime should reproduce all outputs using **Run all** with no manual parameter choices.

### Chapter question

Chapter 3 asks what the classical Womersley solution predicts under harmonic pulsatile forcing and what degrees of freedom it excludes.

This notebook therefore has four computational responsibilities:

1. reproduce the exact single-harmonic Womersley velocity solution and its radial amplitude/phase structure;
2. compute the complex flow-rate, wall-shear, and hydraulic-impedance responses;
3. verify the low- and high-Womersley limits numerically;
4. use VascuQuest waveforms to construct rigid-Womersley reference reconstructions across physiological arterial conditions.

The VascuQuest calculations are **model projections into the Chapter 3 reference state**. They are not claims that PWDB itself is rigid-wall Womersley flow.

### VascuQuest representation

For each virtual subject and arterial site, VascuQuest supplies:

- age;
- heart rate;
- flow-velocity waveform;
- luminal-area waveform.

The source signals are combined internally to recover the book quantity

$$
Q(t)=U(t)A(t).
$$

The time-mean luminal area supplies the rigid reference radius

$$
R=\sqrt{\frac{\langle A\rangle_t}{\pi}}.
$$

With

$$
\Omega=\frac{2\pi}{T},
\qquad
\alpha=R\sqrt{\frac{\Omega}{\nu}},
$$

the subject/site waveform is mapped into the rigid-Womersley reference problem.

For each positive harmonic,

$$
\Omega_m=m\Omega,
\qquad
\alpha_m=\sqrt m\,\alpha,
\qquad
\Lambda_{W,m}=i^{3/2}\alpha_m.
$$

The observed complex flow-rate harmonic $\widehat Q_m$ can be mapped through the Chapter 3 impedance

$$
Z_Q(\Omega_m)=\frac{\widehat G_m}{\widehat Q_m}
$$

to obtain the **reference-model pressure-gradient harmonic**

$$
\widehat G_m=Z_Q(\Omega_m)\widehat Q_m.
$$

That inferred $\widehat G_m$ is then used in the exact Womersley transfer functions for $\widehat u_m(r)$ and $\widehat\tau_{w,m}$.

This construction is deterministic and self-consistent with Chapter 3, but the inferred pressure gradient and wall shear are model-derived reference quantities, not direct PWDB measurements.

In [ ]:
# Configuration and reproducibility constants
from pathlib import Path
import sys, json, subprocess, zipfile

ROOT = Path("/content/nonlinear_arterial_hemodynamics_ch03")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
META_DIR = ROOT / "metadata"
for d in (ROOT, FIG_DIR, DATA_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

VQ_REPOSITORY = "https://github.com/KNOWDYN/VascuQuest.git"
VQ_GIT_REF = "8307147d72e7a6f3ea3135895bd6f52927c67439"
PWDB_RECORD_ID = "3275625"
PWDB_DOI = "10.5281/zenodo.3275625"

rho = 1060.0       # kg m^-3
mu = 3.5e-3        # Pa s
nu = mu / rho      # m^2 s^-1

SITES = [
    "AorticRoot", "ThorAorta", "AbdAorta",
    "Carotid", "Brachial", "Radial", "Femoral",
]

print("Working directory:", ROOT)
print(f"nu = {nu:.6e} m^2/s")

In [ ]:
# Install the pinned VascuQuest revision.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"git+{VQ_REPOSITORY}@{VQ_GIT_REF}"
])

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.special import jv
import vascuquest as vq

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("VascuQuest:", getattr(vq, "__version__", "version field not exposed"))

In [ ]:
# Acquire and verify only the PWDB artifacts used here.
ARTIFACTS = ["model_configurations", "common_site_waveforms_csv"]
verification = {}

for artifact in ARTIFACTS:
    subprocess.run(
        ["vascuquest", "dataset", "acquire",
         "--artifact", artifact, "--yes", "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verified = subprocess.run(
        ["vascuquest", "dataset", "verify",
         "--artifact", artifact, "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verification[artifact] = json.loads(verified.stdout)

status = subprocess.run(
    ["vascuquest", "dataset", "status", "--format", "json"],
    check=True, text=True, capture_output=True,
)
dataset_status = json.loads(status.stdout)
SOURCE_DIR = Path(dataset_status["managed_paths"]["source"])

(META_DIR / "artifact_verification.json").write_text(
    json.dumps(verification, indent=2), encoding="utf-8"
)
(META_DIR / "dataset_status.json").write_text(
    json.dumps(dataset_status, indent=2), encoding="utf-8"
)

print("Verified PWDB source:", SOURCE_DIR)

In [ ]:
# Open the dataset and establish deterministic subject metadata.
session = vq.open_dataset(source=SOURCE_DIR, offline=True)
assert session.identity.record_id == PWDB_RECORD_ID

age_result = session.get("age")
subject_ids = np.asarray(age_result.coordinates[0].values, dtype=str)
ages = np.asarray(age_result.values, dtype=float)

hr_result = session.get("heart_rate", subjects=subject_ids.tolist())
hr_ids = np.asarray(hr_result.coordinates[0].values, dtype=str)
heart_rates = np.asarray(hr_result.values, dtype=float)
assert np.array_equal(subject_ids, hr_ids)

subject_meta = pd.DataFrame({
    "subject_id": subject_ids,
    "age_years": ages,
    "heart_rate_bpm": heart_rates,
})
subject_meta["subject_number"] = subject_meta["subject_id"].astype(int)
subject_meta = subject_meta.sort_values("subject_number").reset_index(drop=True)

# Deterministic representative subject:
# middle source age stratum and median subject number within it.
source_ages = sorted(subject_meta["age_years"].dropna().unique())
target_age = source_ages[len(source_ages)//2]
age_group = subject_meta.loc[subject_meta["age_years"] == target_age].copy()
age_group = age_group.sort_values("subject_number").reset_index(drop=True)
representative_subject = str(age_group.iloc[len(age_group)//2]["subject_id"])

selection_record = {
    "rule": "middle PWDB source age stratum; median canonical subject number",
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "source_age_strata_years": [float(x) for x in source_ages],
}
(META_DIR / "representative_subject.json").write_text(
    json.dumps(selection_record, indent=2), encoding="utf-8"
)
display(pd.DataFrame([selection_record]))

In [ ]:
# Shared B&W figure style, source mapping, and exact Womersley transfer functions.
WAVE_ZIP = SOURCE_DIR / "PWs_csv.zip"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 9.0,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 7.8,
    "axes.linewidth": 0.75,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
BLACK, DARK, MID, LIGHT = "0.0", "0.28", "0.52", "0.74"

SITE_LABELS = {
    "AorticRoot": "Aortic root",
    "ThorAorta": "Thoracic aorta",
    "AbdAorta": "Abdominal aorta",
    "Carotid": "Carotid",
    "Brachial": "Brachial",
    "Radial": "Radial",
    "Femoral": "Femoral",
}

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def save_figure(fig, stem):
    pdf = FIG_DIR / f"{stem}.pdf"
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(png, dpi=600, bbox_inches="tight", pad_inches=0.03)
    return pdf, png

def _wave_member_name(site_id, source_signal):
    basename = f"PWs_{site_id}_{source_signal}.csv"
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        matches = [name for name in zf.namelist() if Path(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {basename!r}; found {len(matches)}")
    return matches[0]

def load_waveform_matrix(site_id, source_signal):
    # Native database names are confined to this source-mapping function.
    member = _wave_member_name(site_id, source_signal)
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        with zf.open(member, "r") as raw:
            frame = pd.read_csv(raw, low_memory=False)
    ids = np.asarray([str(int(x)) for x in frame.iloc[:, 0].to_numpy()], dtype=str)
    values = frame.iloc[:, 1:].to_numpy(dtype=float)
    return ids, values

def Lambda_W(alpha):
    # Chapter 3 convention: Lambda_W = i^(3/2) alpha, with e^(+i Omega t).
    return np.exp(3j*np.pi/4.0) * np.asarray(alpha)

def H_u(x, Omega, alpha):
    LW = Lambda_W(alpha)
    return (1.0/(1j*Omega*rho)) * (1.0 - jv(0, LW*x)/jv(0, LW))

def H_Q(R, Omega, alpha):
    # Qhat = H_Q Ghat
    LW = Lambda_W(alpha)
    return (np.pi*R**2/(1j*Omega*rho)) * (
        1.0 - 2.0*jv(1, LW)/(LW*jv(0, LW))
    )

def Z_Q(R, Omega, alpha):
    return 1.0 / H_Q(R, Omega, alpha)

def H_tau(R, Omega, alpha):
    # tauhat_w = H_tau Ghat for the wall-on-fluid sign convention.
    lambda_W = Lambda_W(alpha) / R
    LW = Lambda_W(alpha)
    return (mu/(1j*Omega*rho)) * (
        lambda_W * jv(1, LW) / jv(0, LW)
    )

def fft_coefficients_real_signal(y):
    y = np.asarray(y, dtype=float)
    return np.fft.rfft(y) / len(y)

print("Shared Womersley functions ready.")

# Reduction of Navier–Stokes

For the Chapter 3 kinematics,

$$
\mathbf u=u_z(r,t)\mathbf e_z,
\qquad
u_r=u_\theta=0,
\qquad
\frac{\partial}{\partial\theta}=0,
\qquad
\frac{\partial u_z}{\partial z}=0.
$$

The nonlinear convective acceleration vanishes identically and Navier–Stokes reduces to

$$
\rho\frac{\partial u_z}{\partial t}
=
G(t)
+
\mu
\left[
\frac{\partial^2u_z}{\partial r^2}
+
\frac1r\frac{\partial u_z}{\partial r}
\right].
$$

The boundary conditions are

$$
u_z(R,t)=0,
\qquad
\left.\frac{\partial u_z}{\partial r}\right|_{r=0}=0.
$$

The cylindrical differential operators used here follow Appendix B. The Newtonian constitutive law and stress conventions remain those of Appendices C and D.

# Single-harmonic forcing

Let

$$
G(t)=\Re\left\{\widehat G e^{i\Omega t}\right\},
$$

and seek

$$
u_z(r,t)=\Re\left\{\widehat u(r)e^{i\Omega t}\right\}.
$$

Define

$$
\alpha=R\sqrt{\frac{\Omega}{\nu}},
\qquad
x=\frac rR,
\qquad
\Lambda_W=i^{3/2}\alpha.
$$

Then

$$
\widehat u(x)
=
\frac{\widehat G}{i\Omega\rho}
\left[
1-
\frac{J_0(\Lambda_Wx)}{J_0(\Lambda_W)}
\right].
$$

The next figure reproduces the radial amplitude and phase structure directly from this equation.

In [ ]:
# Exact single-harmonic Womersley profiles.
x = np.linspace(0.0, 0.995, 600)  # omit the no-slip point from phase plots
alpha_values = [0.5, 2.0, 5.0, 10.0, 20.0]
styles = [
    (BLACK, "-", None),
    (DARK, "--", None),
    (MID, "-.", None),
    (LIGHT, ":", None),
    ("0.12", (0, (5, 2, 1, 2)), None),
]

Omega_ref = 1.0  # only shape/relative phase is used in this normalized figure

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))

for alpha, (gray, ls, _) in zip(alpha_values, styles):
    Hu = H_u(x, Omega_ref, alpha)

    # Normalize amplitude by the centreline amplitude as in the chapter figure logic.
    amp = np.abs(Hu) / np.abs(Hu[0])
    phase = np.unwrap(np.angle(Hu) - np.angle(Hu[0]))

    axes[0].plot(x, amp, color=gray, linestyle=ls, label=rf"$\alpha={alpha:g}$")
    axes[1].plot(x, phase, color=gray, linestyle=ls, label=rf"$\alpha={alpha:g}$")

axes[0].set_xlabel(r"Normalized radius, $x=r/R$")
axes[0].set_ylabel(r"$A_u(r)/A_u(0)$")
axes[0].set_xlim(0, 1)
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].set_xlabel(r"Normalized radius, $x=r/R$")
axes[1].set_ylabel(r"$\phi_u(r)-\phi_u(0)$ (rad)")
axes[1].set_xlim(0, 1)
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.2)
save_figure(fig, "ch03_velocity_transfer_profiles")
plt.show()

The amplitude panel shows the transition from a nearly parabolic low-$\alpha$ profile to an increasingly plug-like core with a narrow near-wall adjustment.

The phase panel shows a feature absent from the quasi-steady picture: at finite $\alpha$, different radii respond with different phase.

These curves are exact deterministic evaluations of the Chapter 3 transfer function.

# Frequency-response observables

## Velocity transfer function

Write

$$
\widehat u(r)=H_u(r;\Omega)\widehat G,
$$

with

$$
H_u(r;\Omega)
=
\frac{1}{i\Omega\rho}
\left[
1-
\frac{J_0(\Lambda_Wr/R)}{J_0(\Lambda_W)}
\right].
$$

The radial amplitude and phase are

$$
A_u(r)=|H_u(r;\Omega)|,
\qquad
\phi_u(r)=\arg H_u(r;\Omega).
$$

## Flow-rate response

The complex flow-rate amplitude is

$$
\widehat Q
=
\frac{\pi R^2\widehat G}{i\Omega\rho}
\left[
1-
\frac{2J_1(\Lambda_W)}
{\Lambda_WJ_0(\Lambda_W)}
\right].
$$

Define

$$
Z_Q(\Omega)=\frac{\widehat G}{\widehat Q}.
$$

## Wall-shear response

The signed complex wall-shear amplitude is

$$
\widehat\tau_w
=
\frac{\mu\widehat G}{i\Omega\rho}
\frac{\lambda_WJ_1(\Lambda_W)}{J_0(\Lambda_W)}.
$$

The following calculation isolates the amplitude and phase evolution of these transfer functions with $\alpha$.

In [ ]:
# Frequency-response observables as functions of alpha.
# Use R=1 and nu=1 only to create dimensionless transfer ratios.
# The plotted quantities are normalized against their Poiseuille limits,
# so no arbitrary dimensional vessel scale enters the comparison.
alpha_grid = np.logspace(-2, 2, 600)
R_ref = 1.0
nu_ref = 1.0
rho_ref = 1.0
mu_ref = 1.0

def HQ_dimensionless(alpha):
    # With R=nu=rho=1, Omega=alpha^2.
    Omega = alpha**2
    LW = Lambda_W(alpha)
    return (np.pi/(1j*Omega)) * (
        1.0 - 2.0*jv(1, LW)/(LW*jv(0, LW))
    )

def Htau_dimensionless(alpha):
    Omega = alpha**2
    LW = Lambda_W(alpha)
    lambda_W = LW
    return (1.0/(1j*Omega)) * (
        lambda_W*jv(1, LW)/jv(0, LW)
    )

HQ = HQ_dimensionless(alpha_grid)
Ht = Htau_dimensionless(alpha_grid)

# Poiseuille limits for the chosen normalization:
HQ_P = np.pi/8.0
Ht_P = -0.5

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15))

axes[0].loglog(alpha_grid, np.abs(HQ/HQ_P), color=BLACK, linestyle="-",
               label=r"$|\widehat Q/\widehat Q_{\mathrm P}|$")
axes[0].loglog(alpha_grid, np.abs(Ht/Ht_P), color=DARK, linestyle="--",
               label=r"$|\widehat\tau_w/\widehat\tau_{w,\mathrm P}|$")
axes[0].set_xlabel(r"Womersley number, $\alpha$")
axes[0].set_ylabel("Amplitude ratio")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].semilogx(alpha_grid, np.angle(HQ/HQ_P), color=BLACK, linestyle="-",
                 label=r"$\arg(\widehat Q/\widehat Q_{\mathrm P})$")
axes[1].semilogx(alpha_grid, np.angle(Ht/Ht_P), color=DARK, linestyle="--",
                 label=r"$\arg(\widehat\tau_w/\widehat\tau_{w,\mathrm P})$")
axes[1].set_xlabel(r"Womersley number, $\alpha$")
axes[1].set_ylabel("Phase relative to Poiseuille limit (rad)")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch03_flow_and_wss_transfer")
plt.show()

The two observables do not acquire the same amplitude or phase response as $\alpha$ increases. The pressure-gradient/flow relation and the pressure-gradient/wall-shear relation are therefore distinct complex transfer functions.

This is why a pulsatile WSS estimate cannot be obtained by simply applying the steady Poiseuille proportionality to every frequency.

# Asymptotic Womersley limits

## Low-Womersley limit

For $\alpha\ll1$,

$$
\widehat u(r)
\sim
\frac{\widehat G R^2}{4\mu}(1-x^2),
$$

$$
\widehat Q
\sim
\frac{\pi R^4}{8\mu}\widehat G,
$$

and

$$
\widehat\tau_w
\sim
-\frac R2\widehat G.
$$

The Womersley solution must converge to these Poiseuille relations.

## High-Womersley limit

For $\alpha\gg1$,

$$
\delta_W
=
\sqrt{\frac{2\nu}{\Omega}}
=
\frac{\sqrt2R}{\alpha},
$$

and away from the wall,

$$
\widehat u_{\mathrm{core}}
\sim
\frac{\widehat G}{i\Omega\rho}.
$$

The next calculation quantifies both limits instead of treating them only qualitatively.

In [ ]:
# Low-alpha convergence error and high-alpha boundary-layer scaling.
alpha_low = np.logspace(-3, 0, 250)
x_check = np.linspace(0.0, 0.98, 500)

max_profile_error = []
Q_error = []
tau_error = []

for a in alpha_low:
    Omega = a*a  # R=nu=1
    Hu = H_u(x_check, Omega, a)

    # Exact Womersley velocity normalized by the Poiseuille amplitude G R^2/(4 mu).
    u_ratio_to_P = Hu / 0.25
    uP_shape = 1.0 - x_check**2
    max_profile_error.append(np.max(np.abs(u_ratio_to_P - uP_shape)))

    hq = HQ_dimensionless(a)
    ht = Htau_dimensionless(a)
    Q_error.append(abs(hq/HQ_P - 1.0))
    tau_error.append(abs(ht/Ht_P - 1.0))

alpha_high = np.logspace(0, 2, 250)
delta_ratio = np.sqrt(2.0) / alpha_high

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15))

axes[0].loglog(alpha_low, max_profile_error, color=BLACK, linestyle="-",
               label=r"velocity profile")
axes[0].loglog(alpha_low, Q_error, color=DARK, linestyle="--",
               label=r"$\widehat Q$")
axes[0].loglog(alpha_low, tau_error, color=MID, linestyle="-.",
               label=r"$\widehat\tau_w$")
axes[0].set_xlabel(r"Womersley number, $\alpha$")
axes[0].set_ylabel("Relative / normalized error")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].loglog(alpha_high, delta_ratio, color=BLACK)
axes[1].axhline(1.0, color=LIGHT, linestyle=":", linewidth=0.9)
axes[1].set_xlabel(r"Womersley number, $\alpha$")
axes[1].set_ylabel(r"$\delta_W/R$")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch03_asymptotic_limits")
plt.show()

The left panel verifies numerically that the exact Womersley solution recovers the Poiseuille reference as $\alpha\to0$.

The right panel expresses the high-$\alpha$ structure directly through

$$
\frac{\delta_W}{R}=\frac{\sqrt2}{\alpha}.
$$

As $\alpha$ grows, viscous adjustment occupies a progressively smaller fraction of the radius while no slip remains enforced at the wall.

# Multiharmonic reconstruction

For a general periodic pressure-gradient waveform,

$$
G(t)=G_0+
\Re\left\{
\sum_{m=1}^{M}\widehat G_m e^{im\Omega t}
\right\},
$$

the mean component is Poiseuille,

$$
u_0(r)=\frac{G_0}{4\mu}(R^2-r^2),
$$

and harmonic $m$ has

$$
\Omega_m=m\Omega,
\qquad
\alpha_m=\sqrt m\,\alpha,
\qquad
\Lambda_{W,m}=i^{3/2}\alpha_m.
$$

Its velocity amplitude is

$$
\widehat u_m(r)
=
\frac{\widehat G_m}{im\Omega\rho}
\left[
1-
\frac{J_0(\Lambda_{W,m}r/R)}
{J_0(\Lambda_{W,m})}
\right].
$$

The physical signal is

$$
u_z(r,t)
=
u_0(r)+
\Re\left\{
\sum_{m=1}^{M}
\widehat u_m(r)e^{im\Omega t}
\right\}.
$$

The linear problem contains no dynamic transfer of energy between harmonics. Each harmonic is filtered independently.

In [ ]:
# Build subject/site rigid-Womersley reference parameters from VascuQuest.
meta = subject_meta.set_index("subject_id")
population_rows = []

for site in SITES:
    print("Processing", SITE_LABELS[site])
    ids_u, U_matrix = load_waveform_matrix(site, "U")
    ids_a, A_matrix = load_waveform_matrix(site, "A")
    assert np.array_equal(ids_u, ids_a)

    for sid, U_row, A_row in zip(ids_u, U_matrix, A_matrix):
        if sid not in meta.index:
            continue

        valid = np.isfinite(U_row) & np.isfinite(A_row)
        if valid.sum() < 16:
            continue

        U_values = U_row[valid]
        A_values = A_row[valid]
        Q_values = U_values * A_values

        R_value = np.sqrt(np.mean(A_values)/np.pi)
        heart_rate_bpm = float(meta.loc[sid, "heart_rate_bpm"])
        age_years = float(meta.loc[sid, "age_years"])

        T_value = 60.0/heart_rate_bpm
        Omega_value = 2.0*np.pi/T_value
        alpha_value = R_value*np.sqrt(Omega_value/nu)

        population_rows.append({
            "subject_id": sid,
            "age_years": age_years,
            "site": site,
            "R_m": R_value,
            "heart_rate_bpm": heart_rate_bpm,
            "T_s": T_value,
            "Omega_s_inv": Omega_value,
            "alpha": alpha_value,
            "mean_Q_m3_s": float(np.mean(Q_values)),
        })

population_df = pd.DataFrame(population_rows)
population_df.to_csv(DATA_DIR / "ch03_population_reference_parameters.csv", index=False)

display(
    population_df.groupby("site")["alpha"]
    .agg(["count", "median", "min", "max"])
    .reindex(SITES)
    .rename(index=SITE_LABELS)
)

In [ ]:
# VascuQuest population: translate each fundamental alpha into the complex
# rigid-Womersley flow and wall-shear transfer response.
response_rows = []

for row in population_df.itertuples(index=False):
    hq = H_Q(row.R_m, row.Omega_s_inv, row.alpha)
    ht = H_tau(row.R_m, row.Omega_s_inv, row.alpha)

    # Compare exact harmonic transfer with its Poiseuille limit at the same R.
    hq_P = np.pi*row.R_m**4/(8.0*mu)
    ht_P = -row.R_m/2.0

    response_rows.append({
        "subject_id": row.subject_id,
        "age_years": row.age_years,
        "site": row.site,
        "alpha": row.alpha,
        "Q_transfer_amplitude_ratio": abs(hq/hq_P),
        "Q_transfer_phase_rad": np.angle(hq/hq_P),
        "tau_transfer_amplitude_ratio": abs(ht/ht_P),
        "tau_transfer_phase_rad": np.angle(ht/ht_P),
    })

response_df = pd.DataFrame(response_rows)
response_df.to_csv(DATA_DIR / "ch03_population_transfer_response.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15))

axes[0].scatter(
    response_df["alpha"],
    response_df["Q_transfer_amplitude_ratio"],
    s=8, facecolors="none", edgecolors=MID, linewidths=0.5,
    label=r"$\widehat Q$ response",
)
axes[0].scatter(
    response_df["alpha"],
    response_df["tau_transfer_amplitude_ratio"],
    s=8, marker="x", color=DARK, linewidths=0.55,
    label=r"$\widehat\tau_w$ response",
)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(r"Womersley number, $\alpha$")
axes[0].set_ylabel("Amplitude ratio to Poiseuille limit")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].scatter(
    response_df["alpha"],
    response_df["Q_transfer_phase_rad"],
    s=8, facecolors="none", edgecolors=MID, linewidths=0.5,
    label=r"$\widehat Q$ response",
)
axes[1].scatter(
    response_df["alpha"],
    response_df["tau_transfer_phase_rad"],
    s=8, marker="x", color=DARK, linewidths=0.55,
    label=r"$\widehat\tau_w$ response",
)
axes[1].set_xscale("log")
axes[1].set_xlabel(r"Womersley number, $\alpha$")
axes[1].set_ylabel("Phase relative to Poiseuille limit (rad)")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch03_vascuquest_transfer_population")
plt.show()

This population figure is not an empirical fit. Every point is the exact Chapter 3 Womersley transfer function evaluated at the $R$ and $\Omega$ supplied by one PWDB subject/site pair.

The result shows how strongly the same classical model can depart from the quasi-steady Poiseuille response over the physiological range of $\alpha$ represented in the virtual population.

## Representative multiharmonic rigid-Womersley reconstruction

A representative PWDB $Q(t)$ waveform can be used as the prescribed periodic flow signal for the Chapter 3 reference model.

The procedure is:

1. compute $\widehat Q_m$ from the VascuQuest waveform;
2. evaluate $Z_Q(\Omega_m)$;
3. infer the reference pressure-gradient harmonic
   $$
   \widehat G_m=Z_Q(\Omega_m)\widehat Q_m;
   $$
4. compute $\widehat u_m(r)$ and $\widehat\tau_{w,m}$ from the exact transfer functions;
5. reconstruct the physical signals.

The resulting pressure gradient, velocity profile, and WSS are therefore **rigid-Womersley model projections consistent with the supplied flow waveform**.

In [ ]:
# Representative multiharmonic reconstruction at the aortic root.
REP_SITE = "AorticRoot"

ids_u, U_matrix = load_waveform_matrix(REP_SITE, "U")
ids_a, A_matrix = load_waveform_matrix(REP_SITE, "A")
assert np.array_equal(ids_u, ids_a)

idx = np.where(ids_u == representative_subject)[0]
if len(idx) != 1:
    raise RuntimeError("Representative subject not found uniquely in waveform archive.")
idx = int(idx[0])

valid = np.isfinite(U_matrix[idx]) & np.isfinite(A_matrix[idx])
U_values = U_matrix[idx][valid]
A_values = A_matrix[idx][valid]
Q_values = U_values * A_values

R_rep = float(np.sqrt(np.mean(A_values)/np.pi))
hr_rep = float(meta.loc[representative_subject, "heart_rate_bpm"])
T_rep = 60.0/hr_rep
Omega_rep = 2.0*np.pi/T_rep
alpha_rep = R_rep*np.sqrt(Omega_rep/nu)

# Fourier coefficients for Q(t): mean plus positive harmonics.
Q_coeff = fft_coefficients_real_signal(Q_values)
M = min(12, len(Q_coeff)-1)

# numpy coefficients reconstruct a real signal as c0 + 2 Re(sum c_m exp(i m phase)).
# Therefore the book-form complex amplitudes are Qhat_m = 2*c_m.
Q0 = float(np.real(Q_coeff[0]))
Qhat = 2.0*Q_coeff[1:M+1]

m = np.arange(1, M+1)
Omega_m = m*Omega_rep
alpha_m = np.sqrt(m)*alpha_rep

Ghat = np.zeros(M, dtype=complex)
tauhat = np.zeros(M, dtype=complex)

for k, (mm, Om, am, qh) in enumerate(zip(m, Omega_m, alpha_m, Qhat)):
    Ghat[k] = Z_Q(R_rep, Om, am) * qh
    tauhat[k] = H_tau(R_rep, Om, am) * Ghat[k]

# Mean pressure-gradient and mean WSS from Poiseuille.
G0 = 8.0*mu*Q0/(np.pi*R_rep**4)
tau0 = -R_rep*G0/2.0

phase = np.arange(len(Q_values), dtype=float)/len(Q_values)
exp_matrix = np.exp(2j*np.pi*np.outer(phase, m))

Q_recon = Q0 + np.real(exp_matrix @ Qhat)
G_recon = G0 + np.real(exp_matrix @ Ghat)
tau_recon = tau0 + np.real(exp_matrix @ tauhat)

recon_df = pd.DataFrame({
    "phase": phase,
    "Q_source_m3_s": Q_values,
    "Q_reconstructed_m3_s": Q_recon,
    "G_reference_Pa_m": G_recon,
    "tau_w_reference_Pa": tau_recon,
})
recon_df.to_csv(DATA_DIR / "ch03_representative_multiharmonic_reconstruction.csv", index=False)

fig, axes = plt.subplots(3, 1, figsize=(6.3, 5.5), sharex=True)

axes[0].plot(phase, Q_values*1e6, color=LIGHT, linestyle=":",
             label="source")
axes[0].plot(phase, Q_recon*1e6, color=BLACK, linestyle="-",
             label=f"{M}-harmonic reconstruction")
axes[0].set_ylabel(r"$Q$ (mL s$^{-1}$)")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].plot(phase, G_recon, color=BLACK)
axes[1].set_ylabel(r"$G$ (Pa m$^{-1}$)")
clean_axes(axes[1])

axes[2].plot(phase, tau_recon, color=BLACK)
axes[2].set_ylabel(r"$\tau_w$ (Pa)")
axes[2].set_xlabel(r"Normalized time, $t/T$")
clean_axes(axes[2])

fig.tight_layout(h_pad=0.45)
save_figure(fig, "ch03_representative_multiharmonic_reconstruction")
plt.show()

print(f"Representative subject: {representative_subject}")
print(f"R = {R_rep*1e3:.3f} mm")
print(f"heart rate = {hr_rep:.1f} min^-1")
print(f"alpha = {alpha_rep:.3f}")

The top panel checks the harmonic reconstruction against the source $Q(t)$. The lower panels are the pressure-gradient and WSS signals required by the rigid-Womersley model to produce that reconstructed flow waveform.

The distinction in evidentiary status is important:

- $Q(t)$ is reconstructed from VascuQuest source data;
- $G(t)$ is inferred through the Chapter 3 hydraulic impedance;
- $\tau_w(t)$ is then computed from the Chapter 3 wall-shear transfer function.

The latter two are **computed reference-model quantities**, not directly observed PWDB signals.

In [ ]:
# Radial waveform distortion for the same representative multiharmonic case.
x_locations = [0.0, 0.5, 0.8, 0.95]
styles = [
    (BLACK, "-", r"$x=0$"),
    (DARK, "--", r"$x=0.5$"),
    (MID, "-.", r"$x=0.8$"),
    (LIGHT, ":", r"$x=0.95$"),
]

# Mean Poiseuille velocity from Q0.
# Q0 = pi R^2 u_center/2, so u_center = 2 Q0/(pi R^2).
u0_center = 2.0*Q0/(np.pi*R_rep**2)

fig, ax = plt.subplots(figsize=(6.0, 3.4))

for xloc, (gray, ls, label) in zip(x_locations, styles):
    u0 = u0_center*(1.0 - xloc**2)
    uhat = np.zeros(M, dtype=complex)

    for k, (Om, am, gh) in enumerate(zip(Omega_m, alpha_m, Ghat)):
        uhat[k] = H_u(xloc, Om, am) * gh

    u_time = u0 + np.real(exp_matrix @ uhat)
    ax.plot(phase, u_time, color=gray, linestyle=ls, label=label)

ax.set_xlabel(r"Normalized time, $t/T$")
ax.set_ylabel(r"$u_z(r,t)$ (m s$^{-1}$)")
ax.legend(frameon=False, ncol=2)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch03_radial_waveform_phase_distortion")
plt.show()

This figure makes the radial transfer structure visible in the time domain. The waveform does not merely decrease toward the wall; its harmonic mixture and phase change with radius because each harmonic carries its own $\alpha_m$ and each radius has its own complex transfer response.

That behavior is intrinsic to the linear Womersley solution and requires no nonlinear coupling.

In [ ]:
# Age-group comparison of the Chapter 3 transfer response at selected sites.
# Use PWDB source age strata exactly as supplied.
AGE_SITES = ["AorticRoot", "Carotid", "Femoral", "Radial"]
age_summary = (
    response_df.loc[response_df["site"].isin(AGE_SITES)]
    .groupby(["age_years", "site"])
    .agg(
        alpha_median=("alpha", "median"),
        Q_phase_median=("Q_transfer_phase_rad", "median"),
        tau_phase_median=("tau_transfer_phase_rad", "median"),
    )
    .reset_index()
)

styles_age = [
    (BLACK, "-", "o"),
    (DARK, "--", "s"),
    (MID, "-.", "^"),
    (LIGHT, ":", "D"),
]

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15))

for site, (gray, ls, marker) in zip(AGE_SITES, styles_age):
    sub = age_summary.loc[age_summary["site"] == site]
    axes[0].plot(
        sub["age_years"], sub["Q_phase_median"],
        color=gray, linestyle=ls, marker=marker, markersize=4,
        label=SITE_LABELS[site],
    )
    axes[1].plot(
        sub["age_years"], sub["tau_phase_median"],
        color=gray, linestyle=ls, marker=marker, markersize=4,
        label=SITE_LABELS[site],
    )

axes[0].set_xlabel("PWDB source age (years)")
axes[0].set_ylabel(r"Median phase of $\widehat Q$ response (rad)")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].set_xlabel("PWDB source age (years)")
axes[1].set_ylabel(r"Median phase of $\widehat\tau_w$ response (rad)")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.2)
save_figure(fig, "ch03_age_group_transfer_phase")
plt.show()

The age-group comparison is descriptive. It reflects how PWDB's age-dependent radius and heart-rate distributions move virtual subjects through the Chapter 3 Womersley transfer functions.

It must not be interpreted as evidence that age directly causes a particular phase shift. The mechanical dependence remains on the model parameters entering $\alpha$, while age is only a population grouping variable.

# Vorticity and the Lamb-vector identity in the reference state

The Womersley field contains azimuthal vorticity,

$$
\boldsymbol\omega
=
-\frac{\partial u_z}{\partial r}\mathbf e_\theta.
$$

Therefore,

$$
\boldsymbol\ell
=
\mathbf u\times\boldsymbol\omega
=
u_z\frac{\partial u_z}{\partial r}\mathbf e_r.
$$

But

$$
\nabla\left(\frac{u_z^2}{2}\right)
=
\boldsymbol\ell,
$$

so

$$
\nabla\left(\frac{u_z^2}{2}\right)
-
\boldsymbol\ell
=
\mathbf0.
$$

The reference solution can therefore contain a finite velocity–vorticity product while maintaining zero convective acceleration. Chapter 2 established the mechanical meaning of this cancellation; Chapter 3 retains it as a property of the control state.

No new plot is added here because the Chapter 2 notebook already visualizes this identity directly. Repeating it would add notebook redundancy without increasing Chapter 3 understanding.

# Degrees of freedom fixed by construction

The Womersley reference state is closed by explicit restrictions:

1. **Straight, constant-radius geometry.**
2. **Rigid wall.**
3. **Scalar isotropic viscosity.**
4. **Axisymmetry and no swirl.**
5. **Fully developed axial motion.**
6. **Linear harmonic response.**
7. **No second-order mean transport generated by the reference problem.**

The VascuQuest calculations above do not relax these restrictions. They only supply physiological values of $R$, $\Omega$, and $Q(t)$ that are projected into this fixed classical model.

This distinction is essential. The notebook uses VascuQuest to interrogate the **range and consequences of the Chapter 3 model**, not to import mechanisms that belong to later chapters.

# What the reader should learn

1. **Womersley flow is a complex frequency-response problem.** A harmonic pressure gradient produces radius-dependent velocity amplitude and phase.

2. **Flow rate and WSS have different transfer functions.** Their amplitude and phase responses depart differently from the Poiseuille limit as $\alpha$ grows.

3. **The low-$\alpha$ limit is a required recovery test.** The exact Bessel solution converges to the Poiseuille velocity profile, flow rate, and wall shear.

4. **The high-$\alpha$ limit separates core inertia from near-wall viscous adjustment.** The controlling thickness is $\delta_W/R=\sqrt2/\alpha$.

5. **Each harmonic sees a different Womersley number.** Since $\alpha_m=\sqrt m\,\alpha$, multiharmonic waveforms are filtered radially and temporally even though the governing velocity equation remains linear.

6. **Linear superposition does not imply identical waveform shape across radius.** Different harmonics acquire different transfer amplitudes and phases.

7. **VascuQuest places the classical model in a physiological parameter range.** PWDB supplies $R$, $\Omega$, and $Q(t)$; the notebook computes the rigid-Womersley reference response associated with those inputs.

8. **Later mechanisms remain excluded.** Nothing in these calculations introduces anisotropy, geometry-sensitive nonlinear modulation, wall motion, nonlinear inter-harmonic transfer, or steady streaming.

# Chapter-enrichment candidates

The notebook produces seven principal figures.

**Candidate 1 — radial Womersley amplitude and phase profiles.**  
The chapter already contains separate amplitude and phase figures. The notebook version is best used to verify or potentially replace those figures if it is cleaner, not added as a duplicate.

**Candidate 2 — flow-rate and WSS transfer functions versus $\alpha$.**  
Strong book candidate. It directly visualizes a result presently carried mainly by equations and prose: $\widehat Q$ and $\widehat\tau_w$ respond differently in both amplitude and phase.

**Candidate 3 — quantitative asymptotic recovery.**  
Strong candidate if the chapter would benefit from an explicit counterfactual/recovery figure showing convergence to Poiseuille and the shrinking high-$\alpha$ viscous layer.

**Candidate 4 — VascuQuest population transfer-response map.**  
Potential candidate. It shows where a virtual arterial population lies on the exact Chapter 3 transfer curves without changing the model.

**Candidate 5 — representative multiharmonic reconstruction of $Q$, $G$, and $\tau_w$.**  
Strong candidate if the chapter needs a concrete demonstration of how harmonic impedance maps a physiological flow waveform into the rigid-Womersley reference pressure-gradient and WSS signals.

**Candidate 6 — radial waveform phase distortion.**  
Potentially strong because it turns the abstract radial phase response into an immediately visible time-domain effect.

**Candidate 7 — age-group transfer-phase comparison.**  
Primarily notebook material unless the age dependence is especially clear and scientifically useful after execution.

No figure is promoted automatically. Existing Chapter 3 figures should be replaced rather than duplicated when a notebook-derived version serves the same scientific purpose better.

In [ ]:
# Reproducibility record.
manifest = {
    "book": "Nonlinear Arterial Hemodynamics",
    "chapter": 3,
    "chapter_title": "Womersley Flow as the Classical Reference State",
    "vascuquest_git_ref": VQ_GIT_REF,
    "pwdb_record_id": PWDB_RECORD_ID,
    "pwdb_doi": PWDB_DOI,
    "rho_kg_m3": rho,
    "mu_Pa_s": mu,
    "nu_m2_s": nu,
    "sites": SITES,
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "representative_site": REP_SITE,
    "representative_R_m": R_rep,
    "representative_heart_rate_bpm": hr_rep,
    "representative_alpha": alpha_rep,
    "retained_positive_harmonics": int(M),
    "source_age_strata_years": [float(x) for x in source_ages],
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "execution_status": "completed to this cell",
    "qualification": (
        "PWDB provides physiological R, heart rate, and Q(t). "
        "Pressure-gradient and wall-shear waveforms generated in this notebook "
        "are rigid-Womersley reference-model projections, not direct PWDB measurements."
    ),
}
(META_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print(json.dumps(manifest, indent=2))
print("\nGenerated PDF figures:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(" -", path.name)

# Reproducibility record

A successful **Run all** execution writes:

- B&W vector PDF figures and high-resolution PNG previews;
- population reference parameters;
- population transfer-function responses;
- representative multiharmonic reconstruction data;
- VascuQuest/PWDB verification metadata;
- deterministic representative-subject metadata;
- a final reproducibility manifest.

No interactive choices, hidden hand-selection, or manual cell-order dependence are permitted in the released notebook.